In [4]:
import numpy as np

class MyMLPClassifier:
    def __init__(self, hidden_layer_sizes=(100,), activation='relu', alpha=0.0001,
                 learning_rate_init=0.001, max_iter=200, random_state=None, verbose=False):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.weights = []
        self.biases = []

    def _initialize_weights_and_biases(self, input_size, output_size):
        layer_sizes = [input_size] + list(self.hidden_layer_sizes) + [output_size]
        for i in range(1, len(layer_sizes)):
            self.weights.append(np.random.randn(layer_sizes[i - 1], layer_sizes[i]) * 0.01)
            self.biases.append(np.zeros((1, layer_sizes[i])))

    def _activation_function(self, z):
        if self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'tanh':
            return np.tanh(z)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        else:
            raise ValueError("Invalid activation function. Supported activations: 'relu', 'tanh', 'sigmoid'.")

    def _derivative_activation_function(self, z):
        if self.activation == 'relu':
            return np.where(z > 0, 1, 0)
        elif self.activation == 'tanh':
            return 1 - np.tanh(z) ** 2
        elif self.activation == 'sigmoid':
            return np.exp(-z) / (1 + np.exp(-z)) ** 2
        else:
            raise ValueError("Invalid activation function. Supported activations: 'relu', 'tanh', 'sigmoid'.")

    def fit(self, X, y):
        self._initialize_weights_and_biases(X.shape[1], 1)

        y = np.reshape(y, (-1, 1))

        for i in range(self.max_iter):
            #forward propagation
            a = [X]
            for weight, bias in zip(self.weights, self.biases):
                z = np.dot(a[-1], weight) + bias
                a.append(self._activation_function(z))

            # backward propagation
            dz = a[-1] - y
            for j in range(len(self.weights) - 1, -1, -1):
                dw = (1 / X.shape[0]) * np.dot(a[j].T, dz)
                db = (1 / X.shape[0]) * np.sum(dz, axis=0, keepdims=True)
                dz = np.dot(dz, self.weights[j].T) * self._derivative_activation_function(a[j])
                self.weights[j] -= self.learning_rate_init * dw
                self.biases[j] -= self.learning_rate_init * db

            if self.verbose and i % 10 == 0:
                loss = np.mean(np.square(y - a[-1]))
                print(f"Epoch {i}, Loss: {loss}")

    def predict(self, X):
        a = X
        for weight, bias in zip(self.weights, self.biases):
            z = np.dot(a, weight) + bias
            a = self._activation_function(z)

        binary_output = np.where(a >= 0.5, 1, 0)
        return binary_output


In [5]:
import random
import numpy as np


def euclideanDistance(point, data) -> float:
    """
    Euclidean distance between point & data.
    Point has shape (m,), data has shape (n, m), and output is (n,)
    """
    return np.sqrt(np.sum((point - data) ** 2, axis=1))


class MyKMeans:
    def __init__(self, nClusters=8, maxIters=1000):
        self.__centroids = None
        self.__nClusters = nClusters
        self.__maxIters = maxIters

    def fit(self, trainInput):
        trainInput = np.array(trainInput) 
        n_samples = trainInput.shape[0]

       
        self.__centroids = []
        self.__centroids.append(trainInput[random.randint(0, n_samples - 1)])
        
        for _ in range(1, self.__nClusters):
            dists = np.min(
                np.array([euclideanDistance(c, trainInput) for c in self.__centroids]),
                axis=0
            )
            probs = dists / np.sum(dists)
            chosen_index = np.random.choice(n_samples, p=probs)
            self.__centroids.append(trainInput[chosen_index])
        
        self.__centroids = np.array(self.__centroids)

        for iteration in range(self.__maxIters):
            labels = []
            for x in trainInput:
                dists = euclideanDistance(x, self.__centroids)
                labels.append(np.argmin(dists))

            new_centroids = []
            for i in range(self.__nClusters):
                cluster_points = trainInput[np.array(labels) == i]
                if len(cluster_points) == 0:
                    new_centroids.append(self.__centroids[i]) 
                else:
                    new_centroids.append(np.mean(cluster_points, axis=0))

            new_centroids = np.array(new_centroids)

            loss = 0
            for i in range(self.__nClusters):
                cluster_points = trainInput[np.array(labels) == i]
                loss += np.sum(euclideanDistance(cluster_points, self.__centroids[i]) ** 2)

            
            print(f"Iteration {iteration}, Loss: {loss}")

            if np.allclose(self.__centroids, new_centroids):
                break
            self.__centroids = new_centroids

    def evaluate(self, testInput):
        testInput = np.array(testInput)
        centroids = []
        centroidsIndexes = []
        for x in testInput:
            dists = euclideanDistance(x, self.__centroids)
            idx = np.argmin(dists)
            centroids.append(self.__centroids[idx])
            centroidsIndexes.append(idx)
        return centroids, centroidsIndexes


In [6]:
def bag_of_words(train_inputs, test_inputs):
    from sklearn.feature_extraction.text import CountVectorizer
    vectorizer = CountVectorizer()
    
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)
    print('vocab: ', vectorizer.get_feature_names_out()[:10])
    print('features: ', train_features.toarray()[:3][:10])

    train_features.toarray()
    test_features.toarray()

    return train_features, test_features


def tf_idf(train_inputs, test_inputs):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(max_features=50)

    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)

    
    print('vocab from train data: ', vectorizer.get_feature_names_out()[:10])
    print('features: ', train_features.toarray()[:3])

    train_features.toarray()
    test_features.toarray()

    return train_features, test_features


import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

def extractFeaturesDoc2Vec(trainInputs, testInputs, vector_size=100, epochs=20):
    # Crearea setului de date etichetat (TaggedDocument)
    train_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(trainInputs)]
    test_data = [text.split() for text in testInputs]  # Doar cuvintele, fără etichete
    
    # Antrenarea modelului Doc2Vec
    model = Doc2Vec(vector_size=vector_size, window=5, min_count=1, workers=4, epochs=epochs)
    model.build_vocab(train_data)
    model.train(train_data, total_examples=model.corpus_count, epochs=model.epochs)
    
    # Extrage vectorii pentru fiecare document
    trainFeatures = [model.infer_vector(text.split()) for text in trainInputs]
    testFeatures = [model.infer_vector(text) for text in test_data]

    trainFeatures = np.array(trainFeatures)
    testFeatures = np.array(testFeatures)
    
    return trainFeatures, testFeatures


In [7]:
def splitData(inputs,outputs):
    np.random.seed(7)

    no_samples = len(inputs)
    indexes = [i for i in range(no_samples)]
    train_sample = np.random.choice(indexes, int(0.8 * no_samples), replace=False)
    test_sample = [i for i in indexes if i not in train_sample]

    train_inputs = [inputs[i] for i in train_sample]
    train_outputs = [outputs[i] for i in train_sample]
    test_inputs = [inputs[i] for i in test_sample]
    test_outputs = [outputs[i] for i in test_sample]

    return train_inputs, train_outputs, test_inputs, test_outputs

In [8]:
import pandas as pd
def data_reader(filename):
    df = pd.read_csv(filename)
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    subset = df[['Text', 'Sentiment']]

    text = [subset.iat[i, 0] for i in range(len(subset))]
    sentiment = [subset.iat[i, 1] for i in range(len(subset))]
    labels = list(set(sentiment))

    return text, sentiment, labels

In [9]:
from sklearn.cluster import KMeans

def predictTool(trainFeatures, testFeatures, labels, classes):
    unsupervisedClassifier = KMeans(n_clusters=classes,n_init=10, random_state=0)
    unsupervisedClassifier.fit(trainFeatures)
    computedIndexes = unsupervisedClassifier.predict(testFeatures)
    computedOutputs = [labels[val] for val in computedIndexes]
    return computedOutputs

In [10]:
def predictMyKmeans(trainFeatures, testFeatures, labels, classes):
    myUnsupervisedClassifier = MyKMeans(nClusters=classes)
    myUnsupervisedClassifier.fit(trainFeatures)
    centroids, computedIndexes = myUnsupervisedClassifier.evaluate(testFeatures)
    computedOutputs = [labels[val] for val in computedIndexes]
    return computedOutputs, centroids, computedIndexes

def predictMyANN(trainFeatures, testFeatures, labels, classes):
    mlp = MyMLPClassifier(hidden_layer_sizes=(50,), activation='relu', max_iter=500, learning_rate_init=0.01, verbose=True)
    mlp.fit(trainFeatures, labels)
    prediction=mlp.predict(testFeatures)
    TestOutputs = ['negative' if elem == 0 else 'positive' for elem in prediction]
    return TestOutputs

In [11]:
from sklearn import neural_network

def predictSupervised(trainInputs, trainOutputs, testInputs):
    classifier = neural_network.MLPClassifier(hidden_layer_sizes=(25, 40, 20), activation='relu', max_iter=500,
                                              solver='adam',
                                              verbose=1, random_state=1, learning_rate_init=0.01 )
    classifier.fit(trainInputs, trainOutputs)
    computedOutputs = classifier.predict(testInputs)
    return computedOutputs

In [12]:

def predictHybrid(trainInputs, trainOutputs, testInputs, testOutputs): 
    # semi-supervised
    from sklearn.cluster import MiniBatchKMeans
    n = 100  # only 100 inputs will be labeled
    classifier = MiniBatchKMeans(n_clusters=n, random_state=0, batch_size=100)
    classifier.fit(trainInputs[:n], trainOutputs[:n])
    computedOutputs = classifier.predict(testInputs)
    prevAcc = accuracy_score(testOutputs, computedOutputs)

    unsupervisedClassifier = KMeans(n_clusters=n, random_state=0)
    dists = unsupervisedClassifier.fit_transform(trainInputs)  # distance matrix points - centroids
    repIdxs = np.argmin(dists, axis=0)
    repInputs = [trainInputs[i] for i in repIdxs]
    repOutputs = [list(trainOutputs)[i] for i in repIdxs]
    classifier = neural_network.MLPClassifier()
    classifier.fit(repInputs, repOutputs)  # fit with the most representative data
    computedOutputs = classifier.predict(testInputs)
    return computedOutputs, prevAcc

In [15]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


print('\nEmotions')
fp = 'reviews_mixed.csv'
text, sentiment, labels = data_reader(fp)
trainInputs, trainOutputs, testInputs, testOutputs = splitData(text, sentiment)
#trainFeatures, testFeatures =bag_of_words(trainInputs, testInputs)
#trainFeatures, testFeatures = tf_idf(trainInputs, testInputs)
trainFeatures, testFeatures = extractFeaturesDoc2Vec(trainInputs, testInputs)

computedOutputs = predictTool(trainFeatures, testFeatures, labels, len(set(labels)))
print('MyANN')
labels2= [0 if elem == 'negative' else 1 for elem in trainOutputs]
myComputedOutputs = predictMyANN(trainFeatures, testFeatures, labels2, len(set(labels)))

print('Supervised')
supervisedOutput = predictSupervised(trainFeatures, trainOutputs, testFeatures)

hybridOutput, prevAcc = predictHybrid(trainFeatures, trainOutputs, testFeatures, testOutputs)
inverseTestOutputs = ['negative' if elem == 'positive' else 'positive' for elem in testOutputs]
accuracyByTool = accuracy_score(testOutputs, computedOutputs)
accuracyByToolInverse = accuracy_score(inverseTestOutputs, computedOutputs)
print('Accuracy score by tool:', max(accuracyByTool, accuracyByToolInverse))

accuracyByMe = accuracy_score(testOutputs, myComputedOutputs)
accuracyByMeInverse = accuracy_score(inverseTestOutputs, myComputedOutputs)
print('Accuracy score by me:', max(accuracyByMe, accuracyByMeInverse))
print('\n')
print('Accuracy score supervised:', accuracy_score(testOutputs, supervisedOutput))
print('\n')
print('Accuracy score hybrid before KMeans:', prevAcc)
print('\n')
print('Accuracy score hybrid after KMeans:', accuracy_score(testOutputs, hybridOutput))
print('\n')
print('Output computed by tool:  ', computedOutputs)
print('\n')
print('Output computed by me:    ', myComputedOutputs)
print('\n')
print('Output for supervised:    ', list(supervisedOutput))
print('\n')
print('Output for hybrid:        ', list(hybridOutput))
print('\n')
print('Real output:              ', testOutputs)


Emotions
MyANN
Epoch 0, Loss: 0.3212098638412151
Epoch 10, Loss: 0.30240497091467683
Epoch 20, Loss: 0.2870261306366268
Epoch 30, Loss: 0.2744507497586369
Epoch 40, Loss: 0.2641672987447974
Epoch 50, Loss: 0.25575838724733685
Epoch 60, Loss: 0.24888239016320632
Epoch 70, Loss: 0.2432598247027278
Epoch 80, Loss: 0.23866223269476253
Epoch 90, Loss: 0.23490273249865967
Epoch 100, Loss: 0.23182851909375035
Epoch 110, Loss: 0.22931467716623724
Epoch 120, Loss: 0.2272590749678114
Epoch 130, Loss: 0.22557817239595104
Epoch 140, Loss: 0.2242036601514168
Epoch 150, Loss: 0.2230796821479368
Epoch 160, Loss: 0.22216056112401908
Epoch 170, Loss: 0.22140896095598328
Epoch 180, Loss: 0.22079433366620463
Epoch 190, Loss: 0.22029170488978056
Epoch 200, Loss: 0.21988066226915806
Epoch 210, Loss: 0.21954451080296372
Epoch 220, Loss: 0.21926958963552698
Epoch 230, Loss: 0.21904472995033494
Epoch 240, Loss: 0.2188608420005806
Epoch 250, Loss: 0.2187104576231934
Epoch 260, Loss: 0.21858744487230647
Epoch 

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
text_input = ["By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."]

# Preprocesare
tfidfVectorizer=TfidfVectorizer(max_features=50)
trainFeatures = tfidfVectorizer.fit_transform(trainInputs).toarray()

text_input_features = tfidfVectorizer.transform(text_input).toarray()

# Predictii folosind modelele
predicted_tool = predictTool(trainFeatures, text_input_features, labels, len(set(labels)))
predicted_mykmeans = predictMyKmeans(trainFeatures, text_input_features, labels, len(set(labels)))
predicted_supervised = predictSupervised(trainFeatures, trainOutputs, text_input_features)

print('\nSentiment estimat:')
print('Cu KMeans Tool: ',predicted_tool)
print('Cu MyKMeans: ',predicted_mykmeans)
print('Cu ANN (Supervised): ',predicted_supervised)


# AZURE

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

key = "EnR4AULaXGwWtetzCvtPEFSmPncN9TT6grph3FNk2PHfzLjugfswJQQJ99BDAC5RqLJXJ3w3AAAaACOGnRJA"
endpoint = "https://analizate.cognitiveservices.azure.com/"

credential = AzureKeyCredential(key)
client = TextAnalyticsClient(endpoint=endpoint, credential=credential)

documents = [text_input[0]]
response = client.analyze_sentiment(documents=documents)[0]

print("Sentiment AZURE:", response.sentiment)

Iteration 0, Loss: 272.9973106012813
Iteration 1, Loss: 133.49175589840735
Iteration 2, Loss: 132.75420920368083
Iteration 3, Loss: 132.66057129260292
Iteration 4, Loss: 132.61674614484878
Iteration 5, Loss: 132.54652997254414
Iteration 6, Loss: 132.39840168464423
Iteration 7, Loss: 132.31368324146362
Iteration 1, loss = 0.62705244
Iteration 2, loss = 0.61087338
Iteration 3, loss = 0.59885550
Iteration 4, loss = 0.58068168
Iteration 5, loss = 0.55765047
Iteration 6, loss = 0.53276515
Iteration 7, loss = 0.50455436
Iteration 8, loss = 0.47348363
Iteration 9, loss = 0.44261513
Iteration 10, loss = 0.41477235
Iteration 11, loss = 0.38977931
Iteration 12, loss = 0.36723351
Iteration 13, loss = 0.34683875
Iteration 14, loss = 0.32752868
Iteration 15, loss = 0.30816134
Iteration 16, loss = 0.28899165
Iteration 17, loss = 0.27009571
Iteration 18, loss = 0.25098266
Iteration 19, loss = 0.23319916
Iteration 20, loss = 0.21649910
Iteration 21, loss = 0.20140018
Iteration 22, loss = 0.18738567
It